# Temporary inference notebook!

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append(os.path.abspath(f'{os.getcwd()}/../..'))
sys.path.append('/mnt/bn/audio-diffusion/pretrained_models/bytegen/models')
import torch
import pytorch_lightning as pl
from recipes.audio_diffusion.modules.pl_module import DiffusionLitModule
from recipes.audio_diffusion.modules.model_types.crash_unet import UNet
from recipes.audio_diffusion.modules.diffusion_schemes import EDMScheme
import torchaudio
from einops import rearrange
import matplotlib.pyplot as plt

import torchaudio
from samantha.utils.hparams import DotDict

from hyperpyyaml import load_hyperpyyaml


def load_config_from_file(config_fp: str):
    with open(config_fp, "r", encoding="utf-8") as fin:
        hparams = load_hyperpyyaml(fin, overrides="")
    return DotDict(hparams)


def load_model(config_fp, ckpt_path, device):
    cfg = load_config_from_file(config_fp)
    pl_module = cfg.pl_module
    pl_datamodule = cfg.pl_datamodule
    encoder = cfg.encoder_transform.to(device)
    pl_module = pl_module.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        model=cfg.model,
        cuda_transforms=encoder,
        map_location=device,
    ).to(device)
    return pl_module, pl_datamodule, encoder

Load model from checkpoint (model type must be set manually atm)

In [ ]:
ckpt_path = '/mnt/bn/audio-diffusion/logs/281298/trials/2276559/diffusion/DAG_350M/checkpoints/last.ckpt'
config = 'conf/latent_diffusion_reboot/DAG_700M_cross_attn.yaml'
device = 'cuda'
pl_module, pl_datamodule, encoder = load_model(config, ckpt_path, device)
print('Loaded! \n')

In [ ]:
audio, cond = next(iter(pl_datamodule.train_dataloader()))

Generate unconditional output

In [ ]:
num_parallel_examples = 18

output_dir = './test_output/'

initial_latents = torch.randn(num_parallel_examples,256,800 * 1, device = device)
text_cond = torch.randn(num_parallel_examples, 1, 128, device = device)
text_cond = cond.to(device)[0:num_parallel_examples].unsqueeze(1)
print('Generating... \n')
with torch.no_grad():
    result = pl_module.generate_samples(initial_latents,text_cond, guidance_scale = 7.0, num_steps = 100)
    result = encoder.decode(result)
print('Saving... \n')
result = (result / result.abs().max()).detach().cpu() 
for i in range(num_parallel_examples):
    torchaudio.save(output_dir + f'test_{i}.wav', result[i,:,:], 24000)
    torchaudio.save(output_dir + f'ref_{i}.wav', audio[i,:,:], 24000)
    
result = (rearrange(result, 'b d n -> d (b n)')/ result.abs().max()).detach().cpu() 
from IPython.display import Audio
Audio(result, rate=24000)

In [ ]:
result.shape

Generate from wav input

In [ ]:
example_input,_ = torchaudio.load('../../arp.wav')

sample_length = 44100 * 8

if example_input.shape[0] < num_signal_channels:
    example_input = example_input.expand([num_signal_channels,-1])
if example_input.shape[0] > num_signal_channels:
    example_input = example_input[:num_signal_channels,:]
    
if example_input.shape[1] > sample_length:
    example_input = example_input[:,:sample_length]
elif example_input.shape[1] < sample_length:
    example_input = torch.cat([example_input, torch.zeros(num_signal_channels,sample_length - example_input.shape[1])],dim=1)


num_parallel_examples = 1
noise_factor = 0.5
input_factor = 2.5

min_churn = 0.1
max_churn = 1.0
steps = 2

noise = noise_factor * torch.randn(num_parallel_examples,num_signal_channels,sample_length, device = device) / input_factor
initial_latents = input_factor * example_input.reshape(1,num_signal_channels,-1).to(device) * (1.0 + noise)

with torch.no_grad():
    result = demo_scheme.sample(initial_latents, model, num_steps= 200, S_churn = min_churn).detach().cpu()
    for i in range(steps-1):
        churn = min_churn + (max_churn - min_churn) * float((i+1)/steps)
        loop = demo_scheme.sample(initial_latents, model, num_steps= 200, S_churn = churn)
        result = torch.cat([result, loop.detach().cpu()], dim = 0)
result = (rearrange(result, 'b d n -> d (b n)')/ result.abs().max()).detach().cpu()

from IPython.display import Audio
Audio(result, rate=44100)

Generate from tones

In [ ]:
sample_length = 44100 * 1

t = (torch.arange(sample_length) / sample_length).expand([num_signal_channels, sample_length])

sine_1 = torch.sin(t * 100 * 6.28)
sine_2 = torch.sin(t * 200 * 1.5 * 6.28)
saw_1 = torch.fmod(t * 200, 1) -0.5

example_input = sine_1

if example_input.shape[0] < num_signal_channels:
    example_input = example_input.expand([num_signal_channels,-1])
if example_input.shape[0] > num_signal_channels:
    example_input = example_input[:num_signal_channels,:]
    
if example_input.shape[1] > sample_length:
    example_input = example_input[:,:sample_length]
elif example_input.shape[1] < sample_length:
    example_input = torch.cat([example_input, torch.zeros(num_signal_channels,sample_length - example_input.shape[1])],dim=1)


num_parallel_examples = 1
noise_factor = 0.01
input_factor = 0.25

noise = noise_factor * torch.randn(num_parallel_examples,num_signal_channels,sample_length, device = device) / input_factor
initial_latents = input_factor * example_input.reshape(1,num_signal_channels,-1).to(device) * (1.0 + noise)

with torch.no_grad():
    result = demo_scheme.sample(initial_latents, model, num_steps= 100, S_churn=0.25)
result = (rearrange(result, 'b d n -> d (b n)')/ result.abs().max()).detach().cpu()

from IPython.display import Audio
Audio(result, rate=44100)

Guided diffusion

In [ ]:
from recipes.audio_diffusion.modules.guidance_models.guidance_wrapper import GuidanceWrapper
from recipes.audio_diffusion.modules.guidance_models.pitch_detector import AverageFreqDetector
from recipes.audio_diffusion.modules.guidance_models.envelope_detector import EnvelopeDetector, calculateEnvelope

num_parallel_examples = 1
sample_length = 44100 * 2

guidance_target = calculateEnvelope([num_parallel_examples, num_signal_channels, sample_length], 0.1).to(device)
guidance_model = GuidanceWrapper(EnvelopeDetector(sample_rate = 44100, response_time_ms = 200)
                                ,guidance_target
                                ,10).to(device)

initial_latents = torch.randn(num_parallel_examples,num_signal_channels,sample_length, device = device)
with torch.no_grad():
    result = demo_scheme.sample(initial_latents, model, num_steps= 100, S_churn=0.25, guidance_model = guidance_model)
result = (rearrange(result, 'b d n -> d (b n)')/ result.abs().max()).detach().cpu()
from IPython.display import Audio
Audio(result, rate=44100)

In [ ]:
from recipes.audio_diffusion.modules.guidance_models.guidance_wrapper import GuidanceWrapper
from recipes.audio_diffusion.modules.guidance_models.crash_classifier import Classifier

num_parallel_examples = 1
sample_length = 44100

classifier = Classifier()
weights = torch.load('/mnt/bn/janne-research/projects/audio_diffusion/pretrained_models/crash_drum_classifier.pt')['model']
classifier.load_state_dict(weights)

guidance_target = torch.tensor([1.0,0.0,0.0], device = device).view(num_parallel_examples, 3,1)
guidance_model = GuidanceWrapper(classifier
                                ,guidance_target
                                ,50
                                ,loss = torch.nn.BCELoss()).to(device)

initial_latents = torch.randn(num_parallel_examples,num_signal_channels,sample_length, device = device)
with torch.no_grad():
    result = demo_scheme.sample(initial_latents, model, num_steps= 100, S_churn=0.05, guidance_model = guidance_model)
result = (rearrange(result, 'b d n -> d (b n)')/ result.abs().max()).detach().cpu()
from IPython.display import Audio
Audio(result, rate=44100)

Save output 

In [ ]:
import subprocess
import os 

output_dir = './test_output/'
if not os.path.exists(output_dir):
    subprocess.run(['mkdir', output_dir])

torchaudio.save(output_dir + f'test_all.wav', result, 44100)
for i in range(num_parallel_examples):
    torchaudio.save(output_dir + f'test{i}.wav', result[:,(i * sample_length):((i+1)*sample_length)], 44100)

In [ ]:
torchaudio.save(output_dir + f'input.wav', example_input, 44100)

Plot

In [ ]:
plt.plot(result[0,:], alpha = 0.5, label = 'generated')
plt.plot(guidance_target[0,0,:].detach().cpu(), alpha = 0.5, label = 'original')
plt.legend()